# Penelope 원본 행의 RDF·SHACL 검사

이미 분석한 실제 원본의 두 행 `PP3:2`, `PP5:25`로 셀 → RDF → Shape → 보고서 경로를 따라갑니다. 새로운 표본이나 벤치마크가 아닌 실행 예제입니다. 원본 XLSX를 다시 추출하지 않고, 보존된 `source-validation/run-001/data.ttl`에서 두 행과 연결된 참조만 가져왔습니다. 전체 Shape와 선택한 RDF를 이 파일에 담았습니다.

```text
보존된 XLSX 추출 결과 → 보존된 RDF → 두 행과 참조 선택
                                      ↓
                               전체 Shape로 검사
                                      ↓
                           원시 보고서 → 원본 확인 요청
```

## 실행 환경

Python 3.11 이상과 uv를 준비하고 다음 명령으로 노트북을 엽니다. 첫 환경 구성에는 패키지 다운로드가 필요할 수 있습니다. 코드 셀은 위에서 아래로 실행하세요. 실행에는 저장소, 원본 XLSX, API 키가 필요하지 않습니다.

```bash
uv run --with jupyterlab --with ipykernel --with rdflib==7.6.0 --with pyshacl==0.40.1 jupyter lab penelope-source-validation.ipynb
```

저장 출력은 `--execute`로 생성했을 때 실제 Python 커널의 실행 결과입니다. 실행하지 않은 노트북에는 결과가 저장되지 않습니다.

In [1]:
import json
import hashlib
from importlib.metadata import version
from rdflib import Graph, Namespace, RDF, Literal
from pyshacl import validate

P = Namespace("https://example.org/penelope/validation/")
SH = Namespace("http://www.w3.org/ns/shacl#")
for package in ("rdflib", "pyshacl"):
    print(package, version(package))

rdflib 7.6.0
pyshacl 0.40.1


## 출처와 선택 범위

원자료는 **Submerged Arc Welding Open Repository**, DOI [10.5281/zenodo.15083865](https://doi.org/10.5281/zenodo.15083865)이며 [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)을 따릅니다. 저자는 Tapia Suárez, Estefanía Alexandra; Pérez Couñago, Inés; Eike Precker, Christian; Fernández Montenegro, Juan Manuel; Ruiz, Cristina입니다. 아래에는 원자료 메타데이터, 파일 해시와 선택 기준을 표시합니다.

`https://example.org/penelope/validation/`은 이 검사에서 만든 스키마 이름공간입니다. `https://example.org/penelope/source/`와 그 아래 `codebook/`은 출처 행과 코드북 참조용 식별자이며 원자료 발행자의 공식 온톨로지가 아닙니다.

In [2]:
PROVENANCE = {'source_attribution': {'title': 'Submerged Arc Welding Open Repository', 'doi': '10.5281/zenodo.15083865', 'authors': ['Tapia Suárez, Estefanía Alexandra', 'Pérez Couñago, Inés', 'Eike Precker, Christian', 'Fernández Montenegro, Juan Manuel', 'Ruiz, Cristina'], 'version': 'v2 (record landing page; absent in API metadata)', 'url': 'https://zenodo.org/records/15083865', 'license': 'CC-BY-4.0', 'license_url': 'https://creativecommons.org/licenses/by/4.0/', 'metadata_url': 'https://zenodo.org/api/records/15083865', 'metadata_sha256': '6af6d32eb8360e564580d828e9001f6775024fca045e261e0226a4b095e8d8b5', 'extract_license': 'CC-BY-4.0'}, 'frozen_run': 'research/actionable-evidence-1/source-validation/run-001', 'selected_annotation_ids': ['PP3:2', 'PP5:25'], 'input_sha256': {'input.json': 'd4a68588bd5841c46d3503f23454956c090626c8216d0b955bec21df67276740', 'data.ttl': 'a4356180e97e02555c12e80d4771a0ae22a7b5db1d5381b02b79f823258b15c0', 'shapes.ttl': 'd7d6c694a13ac6051b347edbc3d7d7638b7b0e63b9a020a46dd6d396053ad423', 'manifest.json': '8f1f9565a6b10f15210525bb66e45c8421e66100608cd8afe6795870c609b618', 'requests.json': 'bd5c8222a9c14b7c62b5287fa2b4bd7a34f86b85f19a633b26087e206fc9394a'}, 'selected_rdf_sha256': 'b549224804b42b5dc79e3c85b5643e683a0be4a737b10fa649379c409e9c6060', 'selection': 'Two annotation subjects and transitively outgoing URI subjects only', 'namespaces': {'schema': 'https://example.org/penelope/validation/', 'source': 'https://example.org/penelope/source/', 'codebook': 'https://example.org/penelope/source/codebook/'}}
print(json.dumps(PROVENANCE, ensure_ascii=False, indent=2))

{
  "source_attribution": {
    "title": "Submerged Arc Welding Open Repository",
    "doi": "10.5281/zenodo.15083865",
    "authors": [
      "Tapia Suárez, Estefanía Alexandra",
      "Pérez Couñago, Inés",
      "Eike Precker, Christian",
      "Fernández Montenegro, Juan Manuel",
      "Ruiz, Cristina"
    ],
    "version": "v2 (record landing page; absent in API metadata)",
    "url": "https://zenodo.org/records/15083865",
    "license": "CC-BY-4.0",
    "license_url": "https://creativecommons.org/licenses/by/4.0/",
    "metadata_url": "https://zenodo.org/api/records/15083865",
    "metadata_sha256": "6af6d32eb8360e564580d828e9001f6775024fca045e261e0226a4b095e8d8b5",
    "extract_license": "CC-BY-4.0"
  },
  "frozen_run": "research/actionable-evidence-1/source-validation/run-001",
  "selected_annotation_ids": [
    "PP3:2",
    "PP5:25"
  ],
  "input_sha256": {
    "input.json": "d4a68588bd5841c46d3503f23454956c090626c8216d0b955bec21df67276740",
    "data.ttl": "a4356180e97e02555c

## 원본 셀과 RDF 참조

선택한 두 Annotation에서 URI 객체를 따라갔습니다. 이렇게 도달한 주어의 트리플을 포함했습니다. 다른 Annotation 행은 포함하지 않았습니다. 셀 주소, 원본 파일 해시, 코드북 행도 함께 읽을 수 있습니다. 아래에는 포함된 RDF 전체와 행별 값을 출력합니다.

`welding_setting`은 코드북의 **명목 설정값**을 가리킵니다. 이 값을 센서로 측정한 실제 전압·전류로 해석하지 않습니다. `expected_defect`와 `expected_welding_setting`은 추출한 코드에서 정한 참조이며 독립적인 정답이 아닙니다.

In [3]:
DATA_TTL = '@prefix p: <https://example.org/penelope/validation/> .\n@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .\n\n<https://example.org/penelope/source/PP3/PP3/2> a p:Annotation ;\n    p:annotationId "PP3:2" ;\n    p:archiveMember "ZENODO_Penelope/PP3/2. ndt_data/defects_xlocation.xlsx" ;\n    p:bead 1 ;\n    p:cell_bead <https://example.org/penelope/source/PP3/PP3/C2> ;\n    p:cell_defect <https://example.org/penelope/source/PP3/PP3/A2> ;\n    p:cell_sample_end <https://example.org/penelope/source/PP3/PP3/G2> ;\n    p:cell_sample_init <https://example.org/penelope/source/PP3/PP3/F2> ;\n    p:cell_welding_setting <https://example.org/penelope/source/PP3/PP3/B2> ;\n    p:cell_x_end_mm <https://example.org/penelope/source/PP3/PP3/E2> ;\n    p:cell_x_init_mm <https://example.org/penelope/source/PP3/PP3/D2> ;\n    p:defect <https://example.org/penelope/source/codebook/PP3%3Adefect_code%3A2> ;\n    p:excelRow 2 ;\n    p:expected_defect <https://example.org/penelope/source/codebook/PP3%3Adefect_code%3A2> ;\n    p:expected_welding_setting <https://example.org/penelope/source/codebook/PP3%3Awelding_settings_code%3A1> ;\n    p:sample_end 196402 ;\n    p:sample_init 49973 ;\n    p:sheet "PP3" ;\n    p:sourceFileSha256 "94236eb7cef2eff8b415c097a2d129ac22c85e0588836ab122e740ae5440a516" ;\n    p:welding_setting <https://example.org/penelope/source/codebook/PP3%3Awelding_settings_code%3A1> ;\n    p:x_end_mm 314.41 ;\n    p:x_init_mm 80 .\n\n<https://example.org/penelope/source/PP5/PP5/25> a p:Annotation ;\n    p:annotationId "PP5:25" ;\n    p:archiveMember "ZENODO_Penelope/PP5/2. ndt_data/defects_xlocation.xlsx" ;\n    p:bead 3 ;\n    p:cell_bead <https://example.org/penelope/source/PP5/PP5/C25> ;\n    p:cell_defect <https://example.org/penelope/source/PP5/PP5/A25> ;\n    p:cell_sample_end <https://example.org/penelope/source/PP5/PP5/G25> ;\n    p:cell_sample_init <https://example.org/penelope/source/PP5/PP5/F25> ;\n    p:cell_welding_setting <https://example.org/penelope/source/PP5/PP5/B25> ;\n    p:cell_x_end_mm <https://example.org/penelope/source/PP5/PP5/E25> ;\n    p:cell_x_init_mm <https://example.org/penelope/source/PP5/PP5/D25> ;\n    p:defect <https://example.org/penelope/source/codebook/PP5%3Adefect_code%3A1> ;\n    p:excelRow 25 ;\n    p:expected_defect <https://example.org/penelope/source/codebook/PP5%3Adefect_code%3A1> ;\n    p:expected_welding_setting <https://example.org/penelope/source/codebook/PP5%3Awelding_settings_code%3A3> ;\n    p:sample_end 66637 ;\n    p:sample_init 73717 ;\n    p:sheet "PP5" ;\n    p:sourceFileSha256 "afa37327d59c76956f4aad85438366b5da0d7475c5d5cc82728eda6c05e89cf9" ;\n    p:welding_setting <https://example.org/penelope/source/codebook/PP5%3Awelding_settings_code%3A3> ;\n    p:x_end_mm 160 ;\n    p:x_init_mm 177 .\n\n<https://example.org/penelope/source/PP3/PP3/A2> p:cellAddress "A2" ;\n    p:sourceFileSha256 "94236eb7cef2eff8b415c097a2d129ac22c85e0588836ab122e740ae5440a516" .\n\n<https://example.org/penelope/source/PP3/PP3/B2> p:cellAddress "B2" ;\n    p:sourceFileSha256 "94236eb7cef2eff8b415c097a2d129ac22c85e0588836ab122e740ae5440a516" .\n\n<https://example.org/penelope/source/PP3/PP3/C2> p:cellAddress "C2" ;\n    p:sourceFileSha256 "94236eb7cef2eff8b415c097a2d129ac22c85e0588836ab122e740ae5440a516" .\n\n<https://example.org/penelope/source/PP3/PP3/D2> p:cellAddress "D2" ;\n    p:sourceFileSha256 "94236eb7cef2eff8b415c097a2d129ac22c85e0588836ab122e740ae5440a516" .\n\n<https://example.org/penelope/source/PP3/PP3/E2> p:cellAddress "E2" ;\n    p:sourceFileSha256 "94236eb7cef2eff8b415c097a2d129ac22c85e0588836ab122e740ae5440a516" .\n\n<https://example.org/penelope/source/PP3/PP3/F2> p:cellAddress "F2" ;\n    p:sourceFileSha256 "94236eb7cef2eff8b415c097a2d129ac22c85e0588836ab122e740ae5440a516" .\n\n<https://example.org/penelope/source/PP3/PP3/G2> p:cellAddress "G2" ;\n    p:sourceFileSha256 "94236eb7cef2eff8b415c097a2d129ac22c85e0588836ab122e740ae5440a516" .\n\n<https://example.org/penelope/source/PP5/PP5/A25> p:cellAddress "A25" ;\n    p:sourceFileSha256 "afa37327d59c76956f4aad85438366b5da0d7475c5d5cc82728eda6c05e89cf9" .\n\n<https://example.org/penelope/source/PP5/PP5/B25> p:cellAddress "B25" ;\n    p:sourceFileSha256 "afa37327d59c76956f4aad85438366b5da0d7475c5d5cc82728eda6c05e89cf9" .\n\n<https://example.org/penelope/source/PP5/PP5/C25> p:cellAddress "C25" ;\n    p:sourceFileSha256 "afa37327d59c76956f4aad85438366b5da0d7475c5d5cc82728eda6c05e89cf9" .\n\n<https://example.org/penelope/source/PP5/PP5/D25> p:cellAddress "D25" ;\n    p:sourceFileSha256 "afa37327d59c76956f4aad85438366b5da0d7475c5d5cc82728eda6c05e89cf9" .\n\n<https://example.org/penelope/source/PP5/PP5/E25> p:cellAddress "E25" ;\n    p:sourceFileSha256 "afa37327d59c76956f4aad85438366b5da0d7475c5d5cc82728eda6c05e89cf9" .\n\n<https://example.org/penelope/source/PP5/PP5/F25> p:cellAddress "F25" ;\n    p:sourceFileSha256 "afa37327d59c76956f4aad85438366b5da0d7475c5d5cc82728eda6c05e89cf9" .\n\n<https://example.org/penelope/source/PP5/PP5/G25> p:cellAddress "G25" ;\n    p:sourceFileSha256 "afa37327d59c76956f4aad85438366b5da0d7475c5d5cc82728eda6c05e89cf9" .\n\n<https://example.org/penelope/source/codebook/PP3%3Adefect_code%3A2> a p:DefectCode ;\n    p:archiveMember "ZENODO_Penelope/PP3/2. ndt_data/defects_xlocation.xlsx" ;\n    p:codeCell "A3" ;\n    p:excelRow 3 ;\n    p:sheet "defect_code" ;\n    p:sourceCode 2 ;\n    p:sourceFileSha256 "94236eb7cef2eff8b415c097a2d129ac22c85e0588836ab122e740ae5440a516" ;\n    p:sourceValuesJSON "[2, \\"Lack of fusion\\"]" .\n\n<https://example.org/penelope/source/codebook/PP3%3Awelding_settings_code%3A1> a p:SettingCode ;\n    p:archiveMember "ZENODO_Penelope/PP3/2. ndt_data/defects_xlocation.xlsx" ;\n    p:codeCell "A3" ;\n    p:excelRow 3 ;\n    p:sheet "welding_settings_code" ;\n    p:sourceCode 1 ;\n    p:sourceFileSha256 "94236eb7cef2eff8b415c097a2d129ac22c85e0588836ab122e740ae5440a516" ;\n    p:sourceValuesJSON "[1, 27, 450, null, null, 1.6, null, 48]" .\n\n<https://example.org/penelope/source/codebook/PP5%3Adefect_code%3A1> a p:DefectCode ;\n    p:archiveMember "ZENODO_Penelope/PP5/2. ndt_data/defects_xlocation.xlsx" ;\n    p:codeCell "A2" ;\n    p:excelRow 2 ;\n    p:sheet "defect_code" ;\n    p:sourceCode 1 ;\n    p:sourceFileSha256 "afa37327d59c76956f4aad85438366b5da0d7475c5d5cc82728eda6c05e89cf9" ;\n    p:sourceValuesJSON "[1, \\"Porosity\\"]" .\n\n<https://example.org/penelope/source/codebook/PP5%3Awelding_settings_code%3A3> a p:SettingCode ;\n    p:archiveMember "ZENODO_Penelope/PP5/2. ndt_data/defects_xlocation.xlsx" ;\n    p:codeCell "A5" ;\n    p:excelRow 5 ;\n    p:sheet "welding_settings_code" ;\n    p:sourceCode 3 ;\n    p:sourceFileSha256 "afa37327d59c76956f4aad85438366b5da0d7475c5d5cc82728eda6c05e89cf9" ;\n    p:sourceValuesJSON "[3, 27, 500, 35, 450, 1.8, 1.95, 72]" .\n\n'
assert hashlib.sha256(DATA_TTL.encode()).hexdigest() == PROVENANCE["selected_rdf_sha256"]
data = Graph().parse(data=DATA_TTL, format="turtle")
print("annotations =", len(set(data.subjects(RDF.type, P.Annotation))))
print("triples =", len(data))
print(DATA_TTL)

annotations = 2
triples = 104
@prefix p: <https://example.org/penelope/validation/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<https://example.org/penelope/source/PP3/PP3/2> a p:Annotation ;
    p:annotationId "PP3:2" ;
    p:archiveMember "ZENODO_Penelope/PP3/2. ndt_data/defects_xlocation.xlsx" ;
    p:bead 1 ;
    p:cell_bead <https://example.org/penelope/source/PP3/PP3/C2> ;
    p:cell_defect <https://example.org/penelope/source/PP3/PP3/A2> ;
    p:cell_sample_end <https://example.org/penelope/source/PP3/PP3/G2> ;
    p:cell_sample_init <https://example.org/penelope/source/PP3/PP3/F2> ;
    p:cell_welding_setting <https://example.org/penelope/source/PP3/PP3/B2> ;
    p:cell_x_end_mm <https://example.org/penelope/source/PP3/PP3/E2> ;
    p:cell_x_init_mm <https://example.org/penelope/source/PP3/PP3/D2> ;
    p:defect <https://example.org/penelope/source/codebook/PP3%3Adefect_code%3A2> ;
    p:excelRow 2 ;
    p:expected_defect <https://example.org/penelope/source/codebook

In [4]:
fields = ("defect", "welding_setting", "bead", "x_init_mm", "x_end_mm", "sample_init", "sample_end")
for annotation_id in PROVENANCE["selected_annotation_ids"]:
    subject = data.value(predicate=P.annotationId, object=Literal(annotation_id))
    print("\n", annotation_id)
    print("source:", data.value(subject, P.archiveMember))
    print("sha256:", data.value(subject, P.sourceFileSha256))
    print("sheet / row:", data.value(subject, P.sheet), data.value(subject, P.excelRow))
    for field in fields:
        cell = data.value(subject, P["cell_" + field])
        value = data.value(subject, P[field])
        print(field, "cell=" + str(data.value(cell, P.cellAddress)), "value=" + str(value))
        if field in ("defect", "welding_setting"):
            print("  sourceCode:", data.value(value, P.sourceCode))
            print("  reference path:", data.value(value, P.archiveMember),
                  data.value(value, P.sheet), data.value(value, P.codeCell))
            print("  codebook row:", data.value(value, P.sourceValuesJSON))


 PP3:2
source: ZENODO_Penelope/PP3/2. ndt_data/defects_xlocation.xlsx
sha256: 94236eb7cef2eff8b415c097a2d129ac22c85e0588836ab122e740ae5440a516
sheet / row: PP3 2
defect cell=A2 value=https://example.org/penelope/source/codebook/PP3%3Adefect_code%3A2
  sourceCode: 2
  reference path: ZENODO_Penelope/PP3/2. ndt_data/defects_xlocation.xlsx defect_code A3
  codebook row: [2, "Lack of fusion"]
welding_setting cell=B2 value=https://example.org/penelope/source/codebook/PP3%3Awelding_settings_code%3A1
  sourceCode: 1
  reference path: ZENODO_Penelope/PP3/2. ndt_data/defects_xlocation.xlsx welding_settings_code A3
  codebook row: [1, 27, 450, null, null, 1.6, null, 48]
bead cell=C2 value=1
x_init_mm cell=D2 value=80
x_end_mm cell=E2 value=314.41
sample_init cell=F2 value=49973
sample_end cell=G2 value=196402

 PP5:25
source: ZENODO_Penelope/PP5/2. ndt_data/defects_xlocation.xlsx
sha256: afa37327d59c76956f4aad85438366b5da0d7475c5d5cc82728eda6c05e89cf9
sheet / row: PP5 25
defect cell=A25 value=h

## 필수 값·참조·구간 순서 조건

아래는 보존된 Shape 전체입니다. `sh:minCount`, `sh:maxCount`, `sh:datatype`, `sh:class`, `sh:equals`로 필수 값, 타입과 코드 참조를 검사합니다. 구간 순서는 다음 부분이 담당합니다. 실제 전체 Shape에는 두 끝점이 존재하고 타입이 유효할 때만 순서 조건을 적용하는 가드도 있습니다.

```turtle
sh:path p:x_init_mm ; sh:lessThanOrEquals p:x_end_mm
sh:path p:sample_init ; sh:lessThanOrEquals p:sample_end
```

이 조건은 연구자가 선언한 `시작 ≤ 끝` 계약입니다. SHACL이 물리적 결함의 진위를 판별하는 것은 아닙니다.

In [5]:
SHAPES_TTL = '@prefix p: <https://example.org/penelope/validation/> .\n@prefix sh: <http://www.w3.org/ns/shacl#> .\n@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .\n\n# SHACL Core only. Order obligations apply only to present, typed pairs.\np:AnnotationShape a sh:NodeShape ; sh:targetClass p:Annotation ;\n sh:property p:defectShape ;\n sh:property p:welding_settingShape ;\n sh:property p:beadShape ;\n sh:property p:x_init_mmShape ;\n sh:property p:x_end_mmShape ;\n sh:property p:sample_initShape ;\n sh:property p:sample_endShape .\n\np:defectShape a sh:PropertyShape ; sh:path p:defect ; sh:minCount 1 ; sh:maxCount 1 ;\n sh:nodeKind sh:IRI ; sh:class p:DefectCode ; sh:equals p:expected_defect .\n\np:welding_settingShape a sh:PropertyShape ; sh:path p:welding_setting ; sh:minCount 1 ; sh:maxCount 1 ;\n sh:nodeKind sh:IRI ; sh:class p:SettingCode ; sh:equals p:expected_welding_setting .\n\np:beadShape a sh:PropertyShape ; sh:path p:bead ; sh:minCount 1 ; sh:maxCount 1 ;\n sh:datatype xsd:integer .\n\np:x_init_mmShape a sh:PropertyShape ; sh:path p:x_init_mm ; sh:minCount 1 ; sh:maxCount 1 ;\n sh:or ( [ sh:datatype xsd:integer ] [ sh:datatype xsd:decimal ] ) .\n\np:x_end_mmShape a sh:PropertyShape ; sh:path p:x_end_mm ; sh:minCount 1 ; sh:maxCount 1 ;\n sh:or ( [ sh:datatype xsd:integer ] [ sh:datatype xsd:decimal ] ) .\n\np:sample_initShape a sh:PropertyShape ; sh:path p:sample_init ; sh:minCount 1 ; sh:maxCount 1 ;\n sh:datatype xsd:integer .\n\np:sample_endShape a sh:PropertyShape ; sh:path p:sample_end ; sh:minCount 1 ; sh:maxCount 1 ;\n sh:datatype xsd:integer .\n\np:x_init_mmOrder a sh:NodeShape ; sh:targetClass p:Annotation ;\n p:normalizedField "x_init_mm" ; p:normalizedKind "order" ;\n sh:or (\n  [ sh:not [ sh:property\n    [ sh:path p:x_init_mm ; sh:minCount 1 ; sh:maxCount 1 ; sh:or ( [ sh:datatype xsd:integer ] [ sh:datatype xsd:decimal ] ) ],\n    [ sh:path p:x_end_mm ; sh:minCount 1 ; sh:maxCount 1 ; sh:or ( [ sh:datatype xsd:integer ] [ sh:datatype xsd:decimal ] ) ] ] ]\n  [ sh:property [ sh:path p:x_init_mm ; sh:lessThanOrEquals p:x_end_mm ] ]\n ) .\n\np:sample_initOrder a sh:NodeShape ; sh:targetClass p:Annotation ;\n p:normalizedField "sample_init" ; p:normalizedKind "order" ;\n sh:or (\n  [ sh:not [ sh:property\n    [ sh:path p:sample_init ; sh:minCount 1 ; sh:maxCount 1 ; sh:datatype xsd:integer ],\n    [ sh:path p:sample_end ; sh:minCount 1 ; sh:maxCount 1 ; sh:datatype xsd:integer ] ] ]\n  [ sh:property [ sh:path p:sample_init ; sh:lessThanOrEquals p:sample_end ] ]\n ) .\n'
assert hashlib.sha256(SHAPES_TTL.encode()).hexdigest() == PROVENANCE["input_sha256"]["shapes.ttl"]
shapes = Graph().parse(data=SHAPES_TTL, format="turtle")
print(SHAPES_TTL)

@prefix p: <https://example.org/penelope/validation/> .
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

# SHACL Core only. Order obligations apply only to present, typed pairs.
p:AnnotationShape a sh:NodeShape ; sh:targetClass p:Annotation ;
 sh:property p:defectShape ;
 sh:property p:welding_settingShape ;
 sh:property p:beadShape ;
 sh:property p:x_init_mmShape ;
 sh:property p:x_end_mmShape ;
 sh:property p:sample_initShape ;
 sh:property p:sample_endShape .

p:defectShape a sh:PropertyShape ; sh:path p:defect ; sh:minCount 1 ; sh:maxCount 1 ;
 sh:nodeKind sh:IRI ; sh:class p:DefectCode ; sh:equals p:expected_defect .

p:welding_settingShape a sh:PropertyShape ; sh:path p:welding_setting ; sh:minCount 1 ; sh:maxCount 1 ;
 sh:nodeKind sh:IRI ; sh:class p:SettingCode ; sh:equals p:expected_welding_setting .

p:beadShape a sh:PropertyShape ; sh:path p:bead ; sh:minCount 1 ; sh:maxCount 1 ;
 sh:datatype xsd:integer .

p:x_init_mmShape a sh

## pySHACL 실행

`validate()`에 두 행의 RDF와 전체 Shape를 전달합니다. `inference="none"`, `advanced=False`로 SHACL Core 조건을 검사합니다. LLM 호출, OWL 추론, SHACL-AF 규칙은 사용하지 않습니다. 적합 여부와 위반 수는 아래 코드의 실행 결과로 확인합니다.

In [6]:
conforms, report, report_text = validate(
    data_graph=data, shacl_graph=shapes, inference="none", advanced=False,
    inplace=False, abort_on_first=False)
print("conforms =", bool(conforms))
results = sorted(report.subjects(RDF.type, SH.ValidationResult), key=str)
print("validation results =", len(results))
for result in results:
    focus = report.value(result, SH.focusNode)
    print("annotation:", data.value(focus, P.annotationId))
    print("source shape:", report.value(result, SH.sourceShape))
    print("constraint:", report.value(result, SH.sourceConstraintComponent))
    print("message:", report.value(result, SH.resultMessage))

conforms = False
validation results = 2
annotation: PP5:25
source shape: https://example.org/penelope/validation/x_init_mmOrder
constraint: http://www.w3.org/ns/shacl#OrConstraintComponent
message: Node <https://example.org/penelope/source/PP5/PP5/25> must conform to one or more shapes in [ sh:not [ sh:property [ sh:maxCount Literal("1", datatype=xsd:integer) ; sh:minCount Literal("1", datatype=xsd:integer) ; sh:or ( [ sh:datatype xsd:integer ] [ sh:datatype xsd:decimal ] ) ; sh:path p:x_end_mm ], [ sh:maxCount Literal("1", datatype=xsd:integer) ; sh:minCount Literal("1", datatype=xsd:integer) ; sh:or ( [ sh:datatype xsd:integer ] [ sh:datatype xsd:decimal ] ) ; sh:path p:x_init_mm ] ] ] , [ sh:property [ sh:lessThanOrEquals p:x_end_mm ; sh:path p:x_init_mm ] ]
annotation: PP5:25
source shape: https://example.org/penelope/validation/sample_initOrder
constraint: http://www.w3.org/ns/shacl#OrConstraintComponent
message: Node <https://example.org/penelope/source/PP5/PP5/25> must conform t

## 원시 보고서

텍스트와 RDF 보고서를 생략 없이 출력합니다. `sh:or` 가드가 있는 순서 조건은 `OrConstraintComponent`로 보고될 수 있습니다. 보고서의 `sourceShape`를 위의 전체 Shape와 대조하세요. 위반 결과 수와 서로 다른 원본 행 수는 구분해야 합니다.

In [7]:
print(report_text)

Validation Report
Conforms: False
Results (2):
Constraint Violation in OrConstraintComponent (http://www.w3.org/ns/shacl#OrConstraintComponent):
	Severity: sh:Violation
	Source Shape: p:sample_initOrder
	Focus Node: <https://example.org/penelope/source/PP5/PP5/25>
	Value Node: <https://example.org/penelope/source/PP5/PP5/25>
	Message: Node <https://example.org/penelope/source/PP5/PP5/25> must conform to one or more shapes in [ sh:not [ sh:property [ sh:datatype xsd:integer ; sh:maxCount Literal("1", datatype=xsd:integer) ; sh:minCount Literal("1", datatype=xsd:integer) ; sh:path p:sample_end ], [ sh:datatype xsd:integer ; sh:maxCount Literal("1", datatype=xsd:integer) ; sh:minCount Literal("1", datatype=xsd:integer) ; sh:path p:sample_init ] ] ] , [ sh:property [ sh:lessThanOrEquals p:sample_end ; sh:path p:sample_init ] ]
Constraint Violation in OrConstraintComponent (http://www.w3.org/ns/shacl#OrConstraintComponent):
	Severity: sh:Violation
	Source Shape: p:x_init_mmOrder
	Focus Node

In [8]:
print(report.serialize(format="turtle"))

@prefix p: <https://example.org/penelope/validation/> .
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

[] a sh:ValidationReport ;
    sh:conforms false ;
    sh:result [ a sh:ValidationResult ;
            sh:focusNode <https://example.org/penelope/source/PP5/PP5/25> ;
            sh:resultMessage "Node <https://example.org/penelope/source/PP5/PP5/25> must conform to one or more shapes in [ sh:not [ sh:property [ sh:maxCount Literal(\"1\", datatype=xsd:integer) ; sh:minCount Literal(\"1\", datatype=xsd:integer) ; sh:or ( [ sh:datatype xsd:integer ] [ sh:datatype xsd:decimal ] ) ; sh:path p:x_end_mm ], [ sh:maxCount Literal(\"1\", datatype=xsd:integer) ; sh:minCount Literal(\"1\", datatype=xsd:integer) ; sh:or ( [ sh:datatype xsd:integer ] [ sh:datatype xsd:decimal ] ) ; sh:path p:x_init_mm ] ] ] , [ sh:property [ sh:lessThanOrEquals p:x_end_mm ; sh:path p:x_init_mm ] ]" ;
            sh:resultSeverity sh:Violation ;
            sh:source

## 원본 확인 요청

보존된 실행에서 `PP5:25` 한 행에 두 구간 순서 위반이 있었습니다. D25→E25는 177→160 mm, F25→G25는 73717→66637입니다. 따라서 이 예제의 후속 조치는 원본 확인 요청입니다. 아래 `request_source_confirmation` 두 건은 연구자가 작성해 보존된 실행에 남긴 요청입니다. 이를 그대로 보여 주며 새 요청 로직을 실행하지 않습니다.

원본 담당자에게 해당 셀의 의미와 값을 확인해야 합니다. 값 교환, 자동 수정, 물리적 수리나 공정 변경은 수행하지 않습니다.

In [9]:
SAVED_REQUESTS = [{'annotation_id': 'PP5:25', 'kind': 'order', 'action': 'request_source_confirmation', 'origin': 'researcher-authored workflow request; not source-issued instruction', 'archive_member': 'ZENODO_Penelope/PP5/2. ndt_data/defects_xlocation.xlsx', 'source_file_sha256': 'afa37327d59c76956f4aad85438366b5da0d7475c5d5cc82728eda6c05e89cf9', 'sheet': 'PP5', 'excel_row': 25, 'fields': {'sample_init': {'value': 73717, 'cell': 'F25'}, 'sample_end': {'value': 66637, 'cell': 'G25'}}, 'instruction': 'Obtain or confirm the cited source evidence; no automatic value change or physical action.'}, {'annotation_id': 'PP5:25', 'kind': 'order', 'action': 'request_source_confirmation', 'origin': 'researcher-authored workflow request; not source-issued instruction', 'archive_member': 'ZENODO_Penelope/PP5/2. ndt_data/defects_xlocation.xlsx', 'source_file_sha256': 'afa37327d59c76956f4aad85438366b5da0d7475c5d5cc82728eda6c05e89cf9', 'sheet': 'PP5', 'excel_row': 25, 'fields': {'x_init_mm': {'value': 177, 'cell': 'D25'}, 'x_end_mm': {'value': 160, 'cell': 'E25'}}, 'instruction': 'Obtain or confirm the cited source evidence; no automatic value change or physical action.'}]
print(json.dumps(SAVED_REQUESTS, ensure_ascii=False, indent=2))

[
  {
    "annotation_id": "PP5:25",
    "kind": "order",
    "action": "request_source_confirmation",
    "origin": "researcher-authored workflow request; not source-issued instruction",
    "archive_member": "ZENODO_Penelope/PP5/2. ndt_data/defects_xlocation.xlsx",
    "source_file_sha256": "afa37327d59c76956f4aad85438366b5da0d7475c5d5cc82728eda6c05e89cf9",
    "sheet": "PP5",
    "excel_row": 25,
    "fields": {
      "sample_init": {
        "value": 73717,
        "cell": "F25"
      },
      "sample_end": {
        "value": 66637,
        "cell": "G25"
      }
    },
    "instruction": "Obtain or confirm the cited source evidence; no automatic value change or physical action."
  },
  {
    "annotation_id": "PP5:25",
    "kind": "order",
    "action": "request_source_confirmation",
    "origin": "researcher-authored workflow request; not source-issued instruction",
    "archive_member": "ZENODO_Penelope/PP5/2. ndt_data/defects_xlocation.xlsx",
    "source_file_sha256": "afa37327d5

## 해석 범위

두 행은 이미 결과를 확인한 원본에서 설명용으로 선택했습니다. 적합 판정은 선언한 조건의 충족 여부이며 물리적 진실, 안전성, 결함 원인, 데이터 완전성을 인증하지 않습니다. 순서 위반도 원자료의 오류임을 확정하지 않습니다. 이 예제는 원본 셀과 검사 결과를 연결하는 경로를 보여 줍니다. JSON/Python으로 같은 조건을 검사할 수 있으며 RDF의 우월성이나 그래프가 반드시 필요하다는 근거는 아닙니다.